# In-silico modelling of therapeutic bispecific antibodies

**From binding kinetics to density-specific antibody–drug conjugate design**  
Virav Shah

This notebook implements a two-compartment pharmacokinetic (PK) model coupled to bispecific cell-surface binding, internalisation and payload-dependent cell kill. I use it to explore how binding kinetics and target expression affect tumour selectivity, then optimise the four binding rates across nine density/potency scenarios.

The starting point is [Stoddard et al. (2026), *Model-optimized bispecific antibodies improve the selectivity of antibody–drug conjugates for tumors*](https://doi.org/10.20935/AcadDrug8407), with binding and PK structure informed by [Sengers et al. (2016)](https://pmc.ncbi.nlm.nih.gov/articles/PMC4968105/) and [Mould et al. (2007)](https://pmc.ncbi.nlm.nih.gov/articles/PMC2000651/).

**Scope.** This is a reference-informed reconstruction with explicitly chosen extensions. The code implements the modelling and optimisation workflow; it does not reproduce every published endpoint or establish experimentally achievable antibody sequences.

### Reading and running

Run the cells in order from a fresh Python kernel. All model code, calibration, sweeps, optimisation, tables and plots are contained here. There are no external model modules, cached search results or required input datasets. The complete run performs more than ten thousand ODE integrations and can take several minutes or longer, depending on the machine. `WORKERS = 1` runs sequentially.

1. Parameters, units and modelling decisions
2. Binding, PK and cell-kill equations
3. Shared physiological calibration
4. Published-candidate comparison
5. Binding-rate and target-density sweeps
6. Multiobjective optimisation and local refinement
7. Selected rates, trajectories and density transfer
8. Sensitivity and interpretation
9. Generate the accompanying LaTeX/PDF report

The last section exports only the report source and the figures it uses. The executed tables and plots remain embedded in this notebook.

In [ ]:
from dataclasses import dataclass, replace, asdict
from pathlib import Path
import importlib.metadata
import itertools
import os
import random
import shutil
import subprocess
import tempfile
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import least_squares, minimize
from joblib import Parallel, delayed, parallel_config
from platypus import Problem, Real, Solution, InjectedPopulation, NSGAII
from IPython.display import display, Markdown

get_ipython().run_line_magic("matplotlib", "inline")

WORKERS = min(4, os.cpu_count() or 1)
GRID_POINTS = 21
SEARCH_EVALUATIONS = 384
SEEDS = (8407, 2026)
ROOT = Path.cwd()
REPORT_DIR = ROOT / "report"
FIGURE_DIR = REPORT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update(
    {
        "figure.dpi": 110,
        "savefig.dpi": 180,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 10,
        "axes.titlesize": 11,
    }
)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 30)


def present_figure(fig, filename=None):
    """Display a figure; export it only when used by the LaTeX report."""
    if filename:
        fig.savefig(FIGURE_DIR / filename, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def parallel_map(function, arguments):
    """Independent simulations in isolated workers, including notebook-defined code."""
    with parallel_config(backend="loky", inner_max_num_threads=1):
        return Parallel(n_jobs=WORKERS)(delayed(function)(arg) for arg in arguments)


versions = pd.DataFrame(
    {
        "Package": ["numpy", "scipy", "pandas", "matplotlib", "platypus-opt", "joblib"],
        "Version": [
            importlib.metadata.version(p)
            for p in [
                "numpy",
                "scipy",
                "pandas",
                "matplotlib",
                "platypus-opt",
                "joblib",
            ]
        ],
    }
)
display(versions)


## 1. Parameters, units and decisions

| Symbol / short form | Meaning | Units |
|---|---|---|
| ADC / bsADC | Antibody–drug conjugate / bispecific ADC | — |
| PK / RO | Pharmacokinetics / receptor occupancy | — / fraction |
| T1, T2 | The two cell-surface targets | — |
| $k_{on,i}$, $k_{off,i}$ | First-arm association and dissociation rates | M$^{-1}$s$^{-1}$, s$^{-1}$ |
| $K_{D,i}=k_{off,i}/k_{on,i}$ | Monovalent equilibrium dissociation constant | M |
| $\ell_1,\ell_2$ | Membrane association rates for binding the second target | dm² mol$^{-1}$ s$^{-1}$ |
| $q_1,q_2$ | Rates of losing T1 or T2 from a doubly bound ADC | s$^{-1}$ |
| $\rho_{ij}$ | Baseline density of target $i$ on population $j$ | mol/dm² |
| $R_{ij},B_{ij},D_j$ | Free target, singly bound ADC, doubly bound ADC | mol/dm² |
| $C_c,C_p$ | Free ADC in central and peripheral compartments | M |
| $V_c,V_p,Q,CL$ | Compartment volumes, distribution flow, background clearance | L, L, L/h, L/h |
| $N_{eff},f_j,a$ | Accessible cell burden, population fraction, surface area per cell | cells, fraction, dm² |
| $V_{eff}=f_VV_{cell}$ | Effective intracellular payload-distribution volume | L/cell |
| $A_j$ | Accumulated intracellular payload concentration | M |
| $k$ | Receptor/complex turnover, interpreted as twice per hour | $2/3600$ s$^{-1}$ |
| DAR | Drug-to-antibody ratio | 1 payload/ADC |
| EC$_{50}$ | Intracellular concentration producing half-maximal kill | 4 µM, except the labelled 10 nM case |
| $E_T,E_H$ | Tumour kill and population-weighted healthy kill | fraction; multiply by 100 for % |

### Choices that complete the model

- **Absolute capacity:** $N_j=N_{eff}f_j$, with total surface $N_ja$. *Percentages require an absolute scale.*
- **Clearance:** explicit target binding/internalisation plus slow background clearance. *Avoid counting target removal twice.* Mould's empirical target-related saturable elimination is not added on top.
- **Background clearance:** $CL=V_c\ln2/(21\times24)$ L/h. *Represent slow nonspecific antibody removal.* This is our assumption, not Mould's fitted clearance or the terminal half-life of this coupled system.
- **Geometry:** 10 µm spherical diameter from Sengers. *Use a reference-based cell geometry.* The area is 314.159 µm² and whole-cell volume is 0.523599 pL.
- **Intracellular conversion:** fit the effective volume fraction $f_V$ once. *Payload distribution need not fill cells.*
- **Dissociation:** a symmetric detailed-balance closure. *Target labels must not matter.* This is our extension of the reference binding cycle.
- **Turnover:** the same first-order rate for free receptors and complexes, with synthesis $k\rho$. *Maintain the prescribed receptor capacity.*
- **Kill:** instantaneous Emax readout of accumulated payload, with no cell-number feedback. *Retain the paper's simplified endpoint.* No payload efflux, degradation, shedding or bystander effect is included.

The reported population fractions sum to 1.0003 when tumour cells are included. They are retained literally; healthy fractions sum to one. The same physiological parameters apply to every density scenario. Only rates, densities and the explicitly labelled payload potency vary.

In [ ]:
@dataclass(frozen=True)
class Parameters:
    """Physical parameters: seconds, molar concentrations, mol/dm² surface densities."""

    kon1: float = 4.6e5
    koff1: float = 3.4e-4
    kon2: float = 4.6e5
    koff2: float = 3.4e-4
    rho1: float = 1e-10
    rho2: float = 1e-10
    k2d1: float = 2.4e13  # binding T1 after T2; Sengers-derived
    k2d2: float = 2.4e13  # binding T2 after T1
    turnover: float = 2 / 3600  # interpretation of "twice-hourly"
    dual_turnover_factor: float = 1.0  # unpublished closure
    ec50: float = 4e-6
    dose: float = 4e-6
    infusion_s: float = 7200
    end_s: float = 86400  # inferred from figure traces
    vc: float = 11.3  # source-reference value, adoption unconfirmed
    vp: float = 41.5
    q_l_h: float = 1.05
    cl_l_h: float = 11.3 * np.log(2) / (21 * 24)  # assumed background clearance
    total_cells: float = 1e13  # initial guess; calibrated before prediction
    area_dm2: float = 4 * np.pi * 5**2 * 1e-10  # 10 micrometre sphere
    cell_volume_l: float = (
        4 * np.pi * 5**3 / 3 * 1e-15
    )  # whole-cell volume; fitted fraction follows
    surface_off_rule: str = "symmetric"  # 'sengers' used only for sensitivity
    mode: str = "bispecific"


NAMES = ("tumor", "healthy_T1", "healthy_T2", "healthy_T12")
FREQ = np.array([0.0003, 0.1, 0.1, 0.01])
HEALTHY_WEIGHTS = np.array([0.1, 0.1, 0.01])  # no-target remainder 0.79
SCALE_DENSITY = 1e-10
SCALE_CONC = 1e-6


RATES = ("kon1", "koff1", "kon2", "koff2")
AREA = 4 * np.pi * 5**2 * 1e-10
VOLUME = 4 * np.pi * 5**3 / 3 * 1e-15
BOUNDS = [(2.0, 7.0), (-5.0, 2.0), (2.0, 7.0), (-5.0, 2.0)]
LIMIT = 0.0095

population_table = pd.DataFrame(
    {
        "Population": [
            "Tumour",
            "Healthy T1",
            "Healthy T2",
            "Healthy T1+T2",
            "Healthy neither",
        ],
        "Frequency (%)": [0.03, 10, 10, 1, 79],
        "T1 / tumour T1": [1, 0.25, 0, 0.25, 0],
        "T2 / tumour T2": [1, 0, 0.25, 0.25, 0],
    }
)
display(population_table)


## 2. Binding, pharmacokinetics and cell kill

First-arm binding and second-arm cross-linking are separate processes:

\[
v_{ij}=k_{on,i}C_cR_{ij}-k_{off,i}B_{ij},\qquad
w_{1j}=\ell_1B_{2j}R_{1j}-q_1D_j,\quad
w_{2j}=\ell_2B_{1j}R_{2j}-q_2D_j.
\]

We share the detailed-balance correction between both arms:

\[
r=\frac{k_{on,2}\ell_1}{k_{on,1}\ell_2},\qquad
q_1=k_{off,1}\sqrt r,\qquad q_2=k_{off,2}/\sqrt r.
\]

This preserves microscopic reversibility and $q_1q_2=k_{off,1}k_{off,2}$. Exchanging both target labels leaves predictions unchanged. The original Sengers convention is retained as an alternative for sensitivity analysis.

For each target-expressing population:

\[
\begin{aligned}
\dot R_{1j}&=k(\rho_{1j}-R_{1j})-v_{1j}-w_{1j}, &
\dot R_{2j}&=k(\rho_{2j}-R_{2j})-v_{2j}-w_{2j},\\
\dot B_{1j}&=v_{1j}-w_{2j}-kB_{1j}, &
\dot B_{2j}&=v_{2j}-w_{1j}-kB_{2j},\\
\dot D_j&=w_{1j}+w_{2j}-kD_j, &
\dot A_j&=\frac{a}{V_{eff}}k(B_{1j}+B_{2j}+D_j).
\end{aligned}
\]

The two PK compartments share drug with all populations through the central compartment:

\[
V_c\dot C_c=u(t)-CL\,C_c-Q(C_c-C_p)-\sum_jN_ja(v_{1j}+v_{2j}),\qquad
V_p\dot C_p=Q(C_c-C_p).
\]

Here $u(t)$ is a constant two-hour infusion delivering $4\times10^{-6}$ mol. Flow rates are converted from L/h to L/s in the implementation. Initially receptors equal their prescribed densities; all drug-related states are zero. An additional cumulative elimination state tracks $\dot M_{elim}=CL\,C_c$.

Finally,

\[
RO_{ij}=\frac{B_{ij}+D_j}{\rho_{ij}},\quad
E_j=\frac{A_j}{EC_{50}+A_j},\quad
E_H=0.10E_{H1}+0.10E_{H2}+0.01E_{H12}.
\]

Healthy kill is calculated **after applying Emax separately to each population**. No-target cells contribute zero. The monospecific comparator retains bivalent binding to one antigen, including its two-arm capture/dissociation factors; simply deleting the second arm would be a different molecule.

The physical equations use M and mol/dm². Solver states are scaled to µM and $10^{-10}$ mol/dm² for numerical conditioning. Integration is split at the infusion end. BDF handles the widely separated kinetic timescales.

In [ ]:
def densities(p):
    """Surface densities for tumour, T1-only, T2-only and dual-positive healthy cells."""
    return np.array(
        [
            [p.rho1, p.rho2],
            [0.25 * p.rho1, 0],
            [0, 0.25 * p.rho2],
            [0.25 * p.rho1, 0.25 * p.rho2],
        ]
    )


def initial(p):
    # Cc,Cp (uM), eliminated (umol); then R1,R2,B1,B2,D,I for each population
    y = np.zeros(27)
    y[3:].reshape(4, 6)[:, :2] = densities(p) / SCALE_DENSITY
    if p.mode == "monospecific":
        y[3:].reshape(4, 6)[:, 1] = 0
    return y


def rhs(t, y, p, infusing):
    """Coupled 27-state PK, receptor binding and payload accumulation equations."""
    cc, cp = y[:2] * SCALE_CONC
    z = y[3:].reshape(4, 6)
    r1, r2, b1, b2, d = (z[:, i] * SCALE_DENSITY for i in range(5))
    out = np.zeros_like(y)
    dz = out[3:].reshape(4, 6)
    k = p.turnover
    kd = k * p.dual_turnover_factor
    rho = densities(p)
    if p.mode == "bispecific":
        # q1 loses T1 from D -> B2; q2 loses T2 from D -> B1.
        q2 = p.koff2
        q1 = (
            p.koff1
            if p.surface_off_rule == "same_bond"
            else p.koff1 * p.kon2 * p.k2d1 / (p.kon1 * p.k2d2)
        )
        if p.surface_off_rule == "symmetric":
            # Detailed balance with the log correction shared equally by both arms.
            # Preserves q1*q2=koff1*koff2 and invariance to exchanging target labels.
            correction = np.sqrt(p.kon2 * p.k2d1 / (p.kon1 * p.k2d2))
            q1, q2 = p.koff1 * correction, p.koff2 / correction
        v1 = p.kon1 * cc * r1 - p.koff1 * b1
        v2 = p.kon2 * cc * r2 - p.koff2 * b2
        w1 = p.k2d1 * b2 * r1 - q1 * d
        w2 = p.k2d2 * b1 * r2 - q2 * d
        dz[:, 0] = k * (rho[:, 0] - r1) - v1 - w1
        dz[:, 1] = k * (rho[:, 1] - r2) - v2 - w2
        dz[:, 2] = v1 - w2 - k * b1
        dz[:, 3] = v2 - w1 - k * b2
        dz[:, 4] = w1 + w2 - kd * d
        uptake = k * (b1 + b2) + kd * d
        exchange = v1 + v2
    else:
        # B1 = singly-bound antibody; D = doubly-bound, identical-arm combinatorics.
        v1 = 2 * p.kon1 * cc * r1 - p.koff1 * b1
        w1 = p.k2d1 * b1 * r1 - 2 * p.koff1 * d
        dz[:, 0] = k * (rho[:, 0] - r1) - v1 - w1
        dz[:, 2] = v1 - w1 - k * b1
        dz[:, 4] = w1 - kd * d
        uptake = k * b1 + kd * d
        exchange = v1
    dz[:, :5] /= SCALE_DENSITY
    dz[:, 5] = p.area_dm2 / p.cell_volume_l * uptake / SCALE_CONC
    # Table 1 frequencies are used literally; sum including tumor is 1.0003.
    surface_area = p.total_cells * FREQ * p.area_dm2
    elim = p.cl_l_h / 3600 * cc
    distribution = p.q_l_h / 3600 * (cc - cp)
    dose_rate = p.dose / p.infusion_s if infusing else 0.0
    out[0] = (
        (dose_rate - elim - distribution - np.dot(surface_area, exchange))
        / p.vc
        / SCALE_CONC
    )
    out[1] = distribution / p.vp / SCALE_CONC
    out[2] = elim / SCALE_CONC
    return out


In [ ]:
def simulate(p=Parameters(), points=121, rtol=1e-7, atol=1e-10, method="BDF"):
    """Integrate one dose and return trajectories, occupancy, kill and drug accounting."""
    if p.surface_off_rule not in ("sengers", "same_bond", "symmetric"):
        raise ValueError("invalid off rule")
    if p.mode not in ("bispecific", "monospecific"):
        raise ValueError("invalid mode")
    if not 0 < p.infusion_s < p.end_s:
        raise ValueError("require 0 < infusion < horizon")
    y0 = initial(p)
    times = np.unique(np.r_[np.linspace(0, p.end_s, points), p.infusion_s])
    ts = []
    ys = []
    for a, b, on in [(0, p.infusion_s, True), (p.infusion_s, p.end_s, False)]:
        sol = solve_ivp(
            lambda t, y: rhs(t, y, p, on),
            (a, b),
            y0,
            method=method,
            rtol=rtol,
            atol=atol,
            dense_output=True,
        )
        if not sol.success:
            raise RuntimeError(sol.message)
        sel = (
            times[(times >= a) & (times <= b)]
            if a == 0
            else times[(times > a) & (times <= b)]
        )
        ts.extend(sel)
        ys.extend(sol.sol(sel).T)
        y0 = sol.y[:, -1]
    y = np.asarray(ys)
    t = np.asarray(ts)
    z = y[:, 3:].reshape(-1, 4, 6)
    # Never clip inside equations; check solver excursions explicitly.
    if y.min() < -1e-6:
        raise RuntimeError(f"negative scaled state {y.min()}")
    intracellular = np.maximum(z[:, :, 5] * SCALE_CONC, 0)
    kills = intracellular / (p.ec50 + intracellular)
    healthy = kills[:, 1:] @ HEALTHY_WEIGHTS
    tumor = kills[:, 0]
    rho = densities(p)
    if p.mode == "bispecific":
        bound = (
            np.stack((z[:, :, 2] + z[:, :, 4], z[:, :, 3] + z[:, :, 4]), axis=-1)
            * SCALE_DENSITY
        )
    else:
        bound = (
            np.stack((z[:, :, 2] + 2 * z[:, :, 4], np.zeros_like(z[:, :, 4])), axis=-1)
            * SCALE_DENSITY
        )
        rho[:, 1] = 0
    occ = np.divide(
        bound, rho[None, :, :], out=np.zeros_like(bound), where=rho[None, :, :] > 0
    )
    surface = (z[:, :, 2] + z[:, :, 3] + z[:, :, 4]) * SCALE_DENSITY
    m_surface = surface @ (p.total_cells * FREQ * p.area_dm2)
    m_internal = intracellular @ (p.total_cells * FREQ * p.cell_volume_l)
    total = (
        y[:, 0] * SCALE_CONC * p.vc
        + y[:, 1] * SCALE_CONC * p.vp
        + y[:, 2] * SCALE_CONC
        + m_surface
        + m_internal
    )
    administered = p.dose * np.minimum(t / p.infusion_s, 1)
    return dict(
        t=t,
        y=y,
        occupancy=occ,
        intracellular=intracellular,
        tumor=tumor,
        healthy=healthy,
        mass_error=total - administered,
        parameters=asdict(p),
    )


def endpoints(p):
    r = simulate(p, points=3)
    return float(r["tumor"][-1]), float(r["healthy"][-1])


SCENARIOS = {
    "figure4_bispecific": Parameters(),
    "figure4_monospecific": Parameters(mode="monospecific"),
    "figure6": Parameters(kon1=1e4, kon2=1e4, koff1=1, koff2=1),
    "figure8": Parameters(kon1=1e4, kon2=5e4, koff1=1, koff2=1, rho1=1e-9, rho2=5e-11),
}


def score(p, tight=False, method="BDF"):
    """Tumour and healthy endpoint fractions used by the optimisation."""
    result = simulate(
        p,
        points=3,
        rtol=1e-9 if tight else 1e-7,
        atol=1e-12 if tight else 1e-10,
        method=method,
    )
    return float(result["tumor"][-1]), float(result["healthy"][-1])


## 3. Calibrate once, then freeze the physiology

Two effective physiological quantities are inferred from approximate visual uptake readings: Figure 4b, 56 µM; Figure 6b, 280 µM, both at 24 hours. We fit accessible cell burden $N_{eff}$ and intracellular volume fraction $f_V$, using log residuals:

\[
\min_{N_{eff},f_V}\sum_{m\in\{4b,6b\}}
\left[\log\frac{A_{T,m}^{model}(24h)}{A_{T,m}^{paper}(24h)}\right]^2.
\]

$10^{10}\le N_{eff}\le10^{14}$ and $0.01\le f_V\le1$ are modelling bounds. The fitted values are effective quantities, not measurements of a patient. These two observations cannot establish physiological identifiability. Their later agreement is **calibration**, not independent validation. Figure 8 is not used in the fit.

In [ ]:
def calibrate():
    """Fit the two shared effective quantities using only the two stated uptake anchors."""
    base = replace(
        Parameters(),
        area_dm2=AREA,
        cell_volume_l=VOLUME,
        cl_l_h=Parameters().vc * np.log(2) / (21 * 24),
        surface_off_rule="symmetric",
    )
    anchors = {"figure4_bispecific": 56.0, "figure6": 280.0}

    def residual(z):
        n, f = 10 ** np.asarray(z)
        return [
            np.log(
                simulate(
                    replace(
                        base,
                        **{k: getattr(SCENARIOS[name], k) for k in RATES},
                        total_cells=n,
                        cell_volume_l=VOLUME * f
                    ),
                    points=3,
                )["intracellular"][-1, 0]
                * 1e6
                / target
            )
            for name, target in anchors.items()
        ]

    fit = least_squares(
        residual,
        [12.9, np.log10(0.5)],
        bounds=([10, -2], [14, 0]),
        xtol=1e-9,
        ftol=1e-9,
        gtol=1e-9,
    )
    if not fit.success:
        raise RuntimeError(fit.message)
    n, f = 10**fit.x
    base = replace(base, total_cells=float(n), cell_volume_l=float(VOLUME * f))
    config = dict(
        status="OUR_REFERENCE_INFORMED_CALIBRATED_MODEL",
        parameters=asdict(base),
        geometry_diameter_um=10.0,
        whole_cell_volume_l=VOLUME,
        effective_intracellular_fraction=float(f),
        calibration_anchors_uM=anchors,
        calibration_residual_log=fit.fun.tolist(),
        calibration_source="Approximate visual 24h uptake readings, not raw data. Fitting is not validation.",
        design_log10_bounds=BOUNDS,
        healthy_design_fraction=LIMIT,
        paper_healthy_limit=0.01,
        decisions={
            "capacity": "N_j = fitted accessible cell count times literal paper population frequency; total surface = N_j * geometric area",
            "clearance": "Explicit binding/internalisation handles target-mediated removal. Background CL=Vc*ln(2)/(21*24) L/h is our assumption, not a Mould fitted parameter or terminal half-life of the two-compartment system.",
            "no_double_counting": "No empirical Mould target-related Michaelis-Menten elimination on top of explicit target removal.",
            "volume": "Effective intracellular fraction of a 10um sphere, fitted once and frozen across densities.",
            "dissociation": "q1=koff1*sqrt(r), q2=koff2/sqrt(r), r=kon2*k2d1/(kon1*k2d2). Symmetric detailed balance is our extension.",
            "search": "Two seeded NSGAII runs then constrained SLSQP. Best verified feasible candidate, not certified global optimum.",
            "domain": "Broad engineering domain, not author-reported bounds or demonstrated engineering reachability.",
            "toxicity": "Weighted Emax at 24h. Nominal limit 0.95% gives numerical headroom, not robustness assurance.",
        },
    )
    return base, config


base, configuration = calibrate()
calibration_table = pd.DataFrame(
    [
        ["Accessible cell burden", base.total_cells, "cells"],
        ["Whole-cell volume", VOLUME * 1e12, "pL"],
        [
            "Effective intracellular fraction",
            configuration["effective_intracellular_fraction"],
            "fraction",
        ],
        ["Effective intracellular volume", base.cell_volume_l * 1e12, "pL"],
        ["Background clearance", base.cl_l_h, "L/h"],
    ],
    columns=["Quantity", "Value", "Unit"],
)
display(calibration_table.style.format({"Value": "{:.6g}"}))


## 4. Published binding-rate candidates: what agrees and what differs

We evaluate the paper's starting bispecific, bivalent monospecific, Figure 6 and Table 2/Figure 8 binding rates **before optimising anything**. Their kinetics and expression densities are fixed as reported; they share the calibrated physiology above.

Figure 6's final tumour occupancy (~35%), Figure 8's tumour uptake (~240 µM), final T2 occupancy (~66%) and tumour kill (~98%) are approximate visual readings, not digitised raw data. The following table reports signed differences and distinguishes these comparisons from the two fitted uptake points. It does not assign a whole-model validation score to a few landmarks.

In [ ]:
published_parameters = {
    name: replace(
        base,
        **{rate: getattr(p, rate) for rate in RATES},
        rho1=p.rho1,
        rho2=p.rho2,
        mode=p.mode
    )
    for name, p in SCENARIOS.items()
}
published_traces = {
    name: simulate(p, points=241, rtol=1e-9, atol=1e-12)
    for name, p in published_parameters.items()
}
baseline_df = pd.DataFrame(
    [
        dict(
            scenario=name,
            tumor_pct=100 * r["tumor"][-1],
            healthy_pct=100 * r["healthy"][-1],
            tumor_uptake_uM=1e6 * r["intracellular"][-1, 0],
            tumor_T2_RO_pct=100 * r["occupancy"][-1, 0, 1],
        )
        for name, r in published_traces.items()
    ]
)
display(baseline_df.round(3))
landmarks = [
    ("Figure 4b", "figure4_bispecific", "tumor_uptake_uM", 56.0, "µM", "Calibration"),
    ("Figure 6b", "figure6", "tumor_uptake_uM", 280.0, "µM", "Calibration"),
    ("Figure 6a", "figure6", "tumor_T2_RO_pct", 35.0, "%", "Not fitted"),
    ("Figure 8b", "figure8", "tumor_uptake_uM", 240.0, "µM", "Not fitted"),
    ("Figure 8a", "figure8", "tumor_T2_RO_pct", 66.0, "%", "Not fitted"),
    ("Figure 8c", "figure8", "tumor_pct", 98.0, "%", "Not fitted"),
]
comparison_rows = []
for figure, scenario, metric, reference, unit, role in landmarks:
    prediction = float(baseline_df.set_index("scenario").loc[scenario, metric])
    comparison_rows.append(
        {
            "Paper panel": figure,
            "Metric": metric,
            "Paper (approx.)": reference,
            "Model": prediction,
            "Difference": prediction - reference,
            "Unit": unit,
            "Relative difference (%)": 100 * (prediction / reference - 1),
            "Role": role,
        }
    )
comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df.round(3))

fig, axes = plt.subplots(3, 3, figsize=(12, 9), layout="constrained")
for row, name in enumerate(["figure4_bispecific", "figure6", "figure8"]):
    r = published_traces[name]
    t = r["t"] / 3600
    for j, label in enumerate(
        ["Tumour (T2)", "Healthy T1 (T1)", "Healthy T2 (T2)", "Healthy dual (T2)"]
    ):
        target = 0 if j == 1 else 1
        axes[row, 0].plot(t, 100 * r["occupancy"][:, j, target], label=label)
    axes[row, 1].plot(t, 1e6 * r["intracellular"][:, 0], label="Tumour")
    healthy_uptake = r["intracellular"][:, 1:] @ HEALTHY_WEIGHTS / 0.21
    axes[row, 1].plot(t, 1e6 * healthy_uptake, label="Healthy target-positive mean")
    axes[row, 2].plot(t, 100 * r["tumor"], label="Tumour")
    axes[row, 2].plot(t, 100 * r["healthy"], label="All healthy")
    for col, ylabel in enumerate(
        ["Occupancy (%)", "Intracellular ADC (µM)", "Cell kill (%)"]
    ):
        axes[row, col].set(
            xlabel="Time (h)", ylabel=ylabel, title=name.replace("_", " "), xlim=(0, 24)
        )
        if row == 0:
            axes[row, col].legend(fontsize=7)
fig.suptitle("Published binding rates evaluated in the reconstructed model")
present_figure(fig, "published_candidates.png")


## 5. Forward parameter sweeps

These grids ask how fixed parameter combinations behave; they are not optimiser output. Binding sweeps vary the specified rates while holding other inputs fixed. The density sweeps follow the paper's two-stage logic: keep the Figure 6 binding rates fixed, vary tumour T1/T2 independently, and update healthy expression to 25% of each corresponding tumour density.

Both density maps use the same new model and frozen calibration as the optimisation below. Their correspondence to Figures 7/S4 is qualitative; no pixel-level agreement with the paper is claimed.

In [ ]:
def evaluate_parameter_set(p):
    return score(p)


def simulate_grid(xs, ys, make_parameters):
    """Return endpoint-percentage arrays with rows=y and columns=x."""
    points = [make_parameters(float(x), float(y)) for y in ys for x in xs]
    values = np.asarray(parallel_map(evaluate_parameter_set, points))
    values = values.reshape(len(ys), len(xs), 2) * 100
    return values[:, :, 0], values[:, :, 1]


on_axis = np.logspace(4, 7, GRID_POINTS)
off_axis = np.logspace(-3, 0, GRID_POINTS)
binding_specs = [
    (
        "Association rates",
        on_axis,
        on_axis,
        "T1 kon (M⁻¹ s⁻¹)",
        "T2 kon (M⁻¹ s⁻¹)",
        lambda x, y: replace(base, kon1=x, kon2=y),
    ),
    (
        "Dissociation rates",
        off_axis,
        off_axis,
        "T1 koff (s⁻¹)",
        "T2 koff (s⁻¹)",
        lambda x, y: replace(base, koff1=x, koff2=y),
    ),
    (
        "Both arms share rates",
        on_axis,
        off_axis,
        "Shared kon (M⁻¹ s⁻¹)",
        "Shared koff (s⁻¹)",
        lambda x, y: replace(base, kon1=x, kon2=x, koff1=y, koff2=y),
    ),
]
binding_grids = []
fig, axes = plt.subplots(2, 3, figsize=(13, 8), layout="constrained")
for col, (name, xs, ys, xlabel, ylabel, make_p) in enumerate(binding_specs):
    tumor, healthy = simulate_grid(xs, ys, make_p)
    binding_grids.append(
        {"name": name, "x": xs, "y": ys, "tumor": tumor, "healthy": healthy}
    )
    for row, values, cmap, label in [
        (0, tumor, "Greens", "Tumour kill (%)"),
        (1, healthy, "Reds", "Healthy kill (%)"),
    ]:
        ax = axes[row, col]
        mesh = ax.pcolormesh(xs, ys, values, shading="nearest", cmap=cmap, vmin=0)
        ax.set(xscale="log", yscale="log", xlabel=xlabel, ylabel=ylabel, title=name)
        fig.colorbar(mesh, ax=ax, label=label)
fig.suptitle("Binding-rate landscapes at equal tumour densities (10⁻¹⁰ mol/dm²)")
present_figure(fig, "binding_landscapes.png")


In [ ]:
fixed_paper_design = replace(base, kon1=1e4, kon2=1e4, koff1=1, koff2=1)
density_grids = []
fig, axes = plt.subplots(2, 2, figsize=(11, 9), layout="constrained")
for row, (name, lo, hi) in enumerate(
    [("Figure 7 range", -12, -8), ("Supplement S4 range", -14, -9)]
):
    axis = np.logspace(lo, hi, GRID_POINTS)
    tumor, healthy = simulate_grid(
        axis, axis, lambda x, y: replace(fixed_paper_design, rho1=x, rho2=y)
    )
    density_grids.append(
        {"name": name, "axis": axis, "tumor": tumor, "healthy": healthy}
    )
    for col, values, cmap, label, levels in [
        (0, tumor, "Greens", "Tumour kill (%)", [15, 30, 45, 60, 75, 90]),
        (1, healthy, "Reds", "Healthy kill (%)", [1.5, 3, 4.5, 6, 7.5]),
    ]:
        ax = axes[row, col]
        mesh = ax.pcolormesh(
            axis,
            axis,
            values,
            shading="nearest",
            cmap=cmap,
            vmin=0,
            vmax=100 if col == 0 else max(8, values.max()),
        )
        visible = [level for level in levels if values.min() < level < values.max()]
        if visible:
            contours = ax.contour(
                axis, axis, values, levels=visible, colors="black", linewidths=0.6
            )
            ax.clabel(contours, fmt="%g", fontsize=8)
        ax.set(
            xscale="log",
            yscale="log",
            xlabel="Tumour T1 (mol/dm²)",
            ylabel="Tumour T2 (mol/dm²)",
            title=name,
        )
        fig.colorbar(mesh, ax=ax, label=label)
fig.suptitle("Density dependence with the published Figure 6 antibody held fixed")
present_figure(fig, "density_maps.png")


## 6. Multiobjective optimisation

The design variables are $\theta=(\log_{10}k_{on,1},\log_{10}k_{off,1},\log_{10}k_{on,2},\log_{10}k_{off,2})$. Every evaluation solves the full time-dependent ODE system.

\[
\min_\theta\left[-E_T(24h),\ E_H(24h)\right],\qquad E_H(24h)\le0.0095.
\]

The final selection rule is **maximum tumour kill among feasible candidates**. We use 0.95% as the nominal healthy-kill limit, leaving numerical headroom below the paper's 1%; it is not a robustness guarantee.

| Setting | Choice |
|---|---|
| Association-rate domain | $10^2$–$10^7$ M$^{-1}$s$^{-1}$ |
| Dissociation-rate domain | $10^{-5}$–$10^2$ s$^{-1}$ |
| Global search | Platypus NSGA-II; population 32; 384 nominal evaluations per seed |
| Independent seeds | 8407 and 2026 |
| Initial population | Symmetric rate grid plus asymmetric candidates |
| Local refinement | Constrained SLSQP from each run's best feasible point; at most 60 iterations |
| Selection | Reintegrate promising candidates at tighter tolerances; select best feasible endpoint |

These domains extend beyond Figure 5's plotted ranges. They are engineering exploration bounds, not experimentally demonstrated reachability. Near-boundary solutions are flagged. A $10^{-7}$ kill-fraction tolerance is allowed around 0.0095; all selected designs must still satisfy the strict 0.01 threshold.

Each density pair receives a separate design. The final scenario changes payload potency to 10 nM; it is not a density-only comparison. Runs with a local iteration limit may still contribute a verified feasible point; termination messages and seed agreement are shown with the results.

In [ ]:
CASES = [
    ("very_low", 1e-13, 1e-13, 4e-6),
    ("low", 1e-12, 1e-12, 4e-6),
    ("medium", 1e-11, 1e-11, 4e-6),
    ("equal_reference", 1e-10, 1e-10, 4e-6),
    ("high", 1e-9, 1e-9, 4e-6),
    ("unequal_10", 1e-9, 1e-10, 4e-6),
    ("unequal_20", 1e-9, 5e-11, 4e-6),
    ("unequal_100", 1e-9, 1e-11, 4e-6),
    ("very_low_potent_payload", 1e-13, 1e-13, 1e-8),
]

case_table = pd.DataFrame(
    CASES, columns=["Scenario", "T1 (mol/dm²)", "T2 (mol/dm²)", "EC50 (M)"]
)
case_table["EC50 (µM)"] = case_table.pop("EC50 (M)") * 1e6
display(case_table)


In [ ]:
def run_case(args):
    """Two seeded multiobjective searches, constrained refinement, and feasible selection."""
    name, rho1, rho2, ec50, base_dict, nfe, seeds = args
    base = replace(Parameters(**base_dict), rho1=rho1, rho2=rho2, ec50=ec50)
    runs = []
    for seed in seeds:
        random.seed(seed)
        np.random.seed(seed)
        cache = {}
        rows = []
        stage = "global"

        def ev(z):
            key = tuple(float(x) for x in z)
            if key not in cache:
                rates = 10 ** np.asarray(z)
                e, h = score(replace(base, **dict(zip(RATES, rates))))
                rows.append(
                    dict(
                        seed=seed,
                        stage=stage,
                        **dict(zip(RATES, map(float, rates))),
                        tumor_kill_pct=100 * e,
                        healthy_kill_pct=100 * h
                    )
                )
                cache[key] = (100 * e, 100 * h)
            return cache[key]

        problem = Problem(4, 2, 1)
        problem.types[:] = [Real(*b) for b in BOUNDS]
        problem.constraints[:] = "<=0"

        def fn(z):
            e, h = ev(z)
            return [-e, h], [h - 100 * LIMIT]

        problem.function = fn
        initial = []
        for on, off in itertools.product(
            [1e2, 1e3, 1e4], [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
        ):
            s = Solution(problem)
            s.variables[:] = np.log10([on, off, on, off])
            initial.append(s)
        for vals in ([1e4, 1, 5e4, 1], [1e2, 1e2, 1e7, 1e-5], [1e7, 1e-5, 1e2, 1e2]):
            s = Solution(problem)
            s.variables[:] = np.log10(vals)
            initial.append(s)
        alg = NSGAII(problem, population_size=32, generator=InjectedPopulation(initial))
        alg.run(nfe)
        feasible = sorted(
            [x for x in rows if x["healthy_kill_pct"] <= 100 * LIMIT],
            key=lambda x: x["tumor_kill_pct"],
            reverse=True,
        )
        local = []
        stage = "local"
        if feasible:
            z0 = np.log10([feasible[0][k] for k in RATES])
            fit = minimize(
                lambda z: -ev(z)[0],
                z0,
                method="SLSQP",
                bounds=BOUNDS,
                constraints=[{"type": "ineq", "fun": lambda z: 100 * LIMIT - ev(z)[1]}],
                options={"maxiter": 60, "ftol": 1e-6, "eps": 1e-3},
            )
            ev(fit.x)
            local.append(
                dict(
                    success=bool(fit.success),
                    message=str(fit.message),
                    iterations=int(fit.nit),
                )
            )
        best = None
        for row in sorted(
            [x for x in rows if x["healthy_kill_pct"] <= 100 * LIMIT + 1e-5],
            key=lambda x: x["tumor_kill_pct"],
            reverse=True,
        ):
            p = replace(base, **{k: row[k] for k in RATES})
            e, h = score(p, tight=True)
            if h <= LIMIT + 1e-7:
                best = dict(
                    row,
                    tumor_kill_pct=100 * e,
                    healthy_kill_pct=100 * h,
                    parameters=asdict(p),
                )
                break

        runs.append(
            dict(
                seed=seed,
                algorithm_nfe=alg.nfe,
                ode_evaluations=len(rows),
                local=local,
                selected=best,
                evaluations=rows,
            )
        )

    selected = max(
        [x["selected"] for x in runs if x["selected"]],
        key=lambda x: x["tumor_kill_pct"],
        default=None,
    )
    result = dict(
        name=name, rho1=rho1, rho2=rho2, ec50=ec50, runs=runs, selected=selected
    )
    return result


In [ ]:
started = time.perf_counter()
arguments = [
    (name, rho1, rho2, ec50, asdict(base), SEARCH_EVALUATIONS, SEEDS)
    for name, rho1, rho2, ec50 in CASES
]
case_results = parallel_map(run_case, arguments)
results = {result["name"]: result for result in case_results}
search_log = pd.DataFrame(
    [
        {
            "Scenario": name,
            "Seed": run["seed"],
            "Fresh ODE evaluations": run["ode_evaluations"],
            "Tumour kill (%)": (
                run["selected"]["tumor_kill_pct"] if run["selected"] else np.nan
            ),
            "Healthy kill (%)": (
                run["selected"]["healthy_kill_pct"] if run["selected"] else np.nan
            ),
            "Local termination": (
                run["local"][0]["message"] if run["local"] else "No feasible start"
            ),
        }
        for name, result in results.items()
        for run in result["runs"]
    ]
)
print(
    f"Completed {len(results)} scenarios in {(time.perf_counter()-started)/60:.1f} minutes."
)
print(
    f"Fresh search/refinement evaluations: {search_log['Fresh ODE evaluations'].sum():,}"
)
display(search_log.round(5))


## 7. Selected designs, trajectories and transfer between densities

For each scenario we compare: (i) the published equal-density rates, (ii) our selected equal-density design transferred without changing its rates, and (iii) a new density-specific design. A transferred candidate that exceeds the healthy threshold is not a feasible success, even if its tumour kill is higher.

The full-precision selected parameters are held in `results`; the displayed rates are rounded for readability. `summary_df` contains the endpoint and rate tables. `selected_traces` contains the full time courses. Search evaluations remain in each run's `evaluations` field.

In [ ]:
selected_parameters = {}
for name, result in results.items():
    if result["selected"] is None:
        raise RuntimeError(
            f"No feasible candidate was found for {name}; do not report an optimum."
        )
    selected_parameters[name] = Parameters(**result["selected"]["parameters"])
selected_traces = {
    name: simulate(p, points=241, rtol=1e-9, atol=1e-12)
    for name, p in selected_parameters.items()
}
equal_design = selected_parameters["equal_reference"]
summary_rows = []
for name, rho1, rho2, ec50 in CASES:
    p = selected_parameters[name]
    result = selected_traces[name]
    original = score(
        replace(fixed_paper_design, rho1=rho1, rho2=rho2, ec50=ec50), tight=True
    )
    transferred = score(
        replace(equal_design, rho1=rho1, rho2=rho2, ec50=ec50), tight=True
    )
    seed_efficacy = [
        run["selected"]["tumor_kill_pct"]
        for run in results[name]["runs"]
        if run["selected"]
    ]
    rates = {rate: getattr(p, rate) for rate in RATES}
    summary_rows.append(
        dict(
            scenario=name,
            rho1=rho1,
            rho2=rho2,
            ec50_uM=ec50 * 1e6,
            **rates,
            KD1_uM=p.koff1 / p.kon1 * 1e6,
            KD2_uM=p.koff2 / p.kon2 * 1e6,
            paper_equal_rates_tumor_pct=100 * original[0],
            paper_equal_rates_healthy_pct=100 * original[1],
            transferred_equal_design_tumor_pct=100 * transferred[0],
            transferred_equal_design_healthy_pct=100 * transferred[1],
            optimized_tumor_pct=100 * result["tumor"][-1],
            optimized_healthy_pct=100 * result["healthy"][-1],
            seed_tumor_spread_pp=max(seed_efficacy) - min(seed_efficacy),
            near_search_boundary=any(
                min(abs(np.log10(rates[k]) - lo), abs(np.log10(rates[k]) - hi)) < 0.03
                for k, (lo, hi) in zip(RATES, BOUNDS)
            ),
        )
    )
summary_df = pd.DataFrame(summary_rows)
display(
    summary_df[
        [
            "scenario",
            "rho1",
            "rho2",
            "ec50_uM",
            "transferred_equal_design_tumor_pct",
            "transferred_equal_design_healthy_pct",
            "optimized_tumor_pct",
            "optimized_healthy_pct",
        ]
    ].style.format(precision=4, formatter={"rho1": "{:.1e}", "rho2": "{:.1e}"})
)
display(
    summary_df[
        ["scenario", *RATES, "KD1_uM", "KD2_uM", "near_search_boundary"]
    ].style.format({c: "{:.5g}" for c in [*RATES, "KD1_uM", "KD2_uM"]})
)


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12, 9), layout="constrained")
for ax, (name, rho1, rho2, ec50) in zip(axes.flat, CASES):
    r = selected_traces[name]
    ax.plot(r["t"] / 3600, 100 * r["tumor"], label="Tumour")
    ax.plot(r["t"] / 3600, 100 * r["healthy"], label="Healthy")
    ax.axhline(1, color="grey", linestyle=":", linewidth=0.7)
    ax.set(
        title=f"T1 {rho1:.0e} / T2 {rho2:.0e}; EC50 {ec50*1e6:g} µM",
        xlabel="Time (h)",
        ylabel="Cell kill (%)",
        xlim=(0, 24),
        ylim=(0, 100),
    )
axes.flat[0].legend()
fig.suptitle("Density-specific designs: direct ODE time courses")
present_figure(fig, "optimized_timecourses.png")

fig, axes = plt.subplots(2, 1, figsize=(12, 7), layout="constrained")
x = np.arange(len(summary_df))
for ax, kind, label in [
    (axes[0], "tumor", "Tumour kill (%)"),
    (axes[1], "healthy", "Healthy kill (%)"),
]:
    ax.bar(
        x - 0.2,
        summary_df[f"transferred_equal_design_{kind}_pct"],
        0.4,
        label="Equal-density design transferred",
    )
    ax.bar(
        x + 0.2,
        summary_df[f"optimized_{kind}_pct"],
        0.4,
        label="Density-specific optimisation",
    )
    ax.set_ylabel(label)
    ax.set_xticks(
        x, [name.replace("_", "\n") for name in summary_df.scenario], fontsize=8
    )
axes[0].legend()
axes[1].axhline(1, color="red", linestyle="--", label="1% threshold")
present_figure(fig, "density_reoptimization_comparison.png")


### The screened design space and observed Pareto fronts

The points below are **different candidate antibodies**, not noisy trajectories. The line joins the observed nondominated feasible candidates: moving along it trades healthy-cell toxicity against tumour-cell efficacy. It is a finite sampled front, not a proven global frontier. Local refinement targets its highest feasible tumour-kill endpoint.

In [ ]:
def observed_front(frame):
    """Empirical nondominated front: maximise tumour kill and minimise healthy kill."""
    candidates = frame.loc[frame.healthy_kill_pct <= 100 * LIMIT + 1e-5].sort_values(
        ["healthy_kill_pct", "tumor_kill_pct"], ascending=[True, False]
    )
    keep, best = [], -np.inf
    for index, row in candidates.iterrows():
        if row.tumor_kill_pct > best:
            keep.append(index)
            best = row.tumor_kill_pct
    return candidates.loc[keep]


fig, axes = plt.subplots(3, 3, figsize=(12, 9), layout="constrained")
for ax, (name, rho1, rho2, ec50) in zip(axes.flat, CASES):
    evaluated = pd.DataFrame(
        [row for run in results[name]["runs"] for row in run["evaluations"]]
    )
    front = observed_front(evaluated)
    ax.scatter(
        evaluated.healthy_kill_pct,
        evaluated.tumor_kill_pct,
        s=6,
        alpha=0.18,
        color="grey",
    )
    ax.plot(
        front.healthy_kill_pct, front.tumor_kill_pct, color="tab:blue", linewidth=1.5
    )
    best = results[name]["selected"]
    ax.scatter(
        best["healthy_kill_pct"],
        best["tumor_kill_pct"],
        marker="*",
        s=80,
        color="tab:red",
        zorder=5,
    )
    ax.axvline(0.95, color="black", linestyle=":", linewidth=0.8)
    ax.set(
        title=name.replace("_", " "),
        xlabel="Healthy kill (%)",
        ylabel="Tumour kill (%)",
        xlim=(0, 2),
        ylim=(0, 100),
    )
fig.suptitle("Screened candidates, observed feasible fronts, and selected designs")
present_figure(fig)


## 8. Sensitivity and interpretation

The selected rates remain fixed while accessible cell burden, effective intracellular volume and turnover are independently changed by ±20%. We also change background clearance and restore the original Sengers dissociation convention. These are diagnostic scenarios, not confidence intervals. A result above 1% healthy kill indicates sensitivity to that assumption; nominal feasibility is not robust safety.

Numerical safeguards remain inside the implementation: solver success, nonnegative states, mass accounting and independent integration of the selected designs. They produce no separate check files. They address computational consistency, not biological validation.

In [ ]:
sensitivity_rows, numerical_rows = [], []
for name, p in selected_parameters.items():
    r = selected_traces[name]
    alternate = score(p, tight=True, method="Radau")
    solver_error = max(
        abs(alternate[0] - r["tumor"][-1]), abs(alternate[1] - r["healthy"][-1])
    )
    mass_error = float(np.max(np.abs(r["mass_error"])) / p.dose)
    swapped = replace(
        p,
        rho1=p.rho2,
        rho2=p.rho1,
        kon1=p.kon2,
        kon2=p.kon1,
        koff1=p.koff2,
        koff2=p.koff1,
        k2d1=p.k2d2,
        k2d2=p.k2d1,
    )
    swap_score = score(swapped, tight=True)
    swap_error = max(
        abs(swap_score[0] - r["tumor"][-1]), abs(swap_score[1] - r["healthy"][-1])
    )
    assert solver_error < 1e-6 and mass_error < 1e-6 and swap_error < 1e-6
    assert np.max(r["healthy"]) < 0.01
    numerical_rows.append(
        dict(
            scenario=name,
            solver_error_fraction=solver_error,
            mass_relative_error=mass_error,
            target_exchange_error_fraction=swap_error,
        )
    )
    variants = {
        "Accessible cells -20%": replace(p, total_cells=0.8 * p.total_cells),
        "Accessible cells +20%": replace(p, total_cells=1.2 * p.total_cells),
        "Effective volume -20%": replace(p, cell_volume_l=0.8 * p.cell_volume_l),
        "Effective volume +20%": replace(p, cell_volume_l=1.2 * p.cell_volume_l),
        "Turnover -20%": replace(p, turnover=0.8 * p.turnover),
        "Turnover +20%": replace(p, turnover=1.2 * p.turnover),
        "Sengers off-rate convention": replace(p, surface_off_rule="sengers"),
        "Zero background clearance": replace(p, cl_l_h=0),
        "Double background clearance": replace(p, cl_l_h=2 * p.cl_l_h),
    }
    for label, variant in variants.items():
        tumor, healthy = score(variant, tight=True)
        sensitivity_rows.append(
            dict(
                scenario=name,
                perturbation=label,
                tumor_kill_pct=100 * tumor,
                healthy_kill_pct=100 * healthy,
            )
        )
sensitivity_df = pd.DataFrame(sensitivity_rows)
numerical_df = pd.DataFrame(numerical_rows)
summary_df["sensitivity_max_healthy_pct"] = summary_df.scenario.map(
    sensitivity_df.groupby("scenario").healthy_kill_pct.max()
)
display(
    summary_df[
        ["scenario", "seed_tumor_spread_pp", "sensitivity_max_healthy_pct"]
    ].round(6)
)
print(
    f"Maximum cross-solver endpoint difference: {numerical_df.solver_error_fraction.max():.2e} (fraction)."
)
print(
    f"Maximum relative drug-mass error: {numerical_df.mass_relative_error.max():.2e}."
)

matrix = sensitivity_df.pivot(
    index="scenario", columns="perturbation", values="healthy_kill_pct"
).reindex(summary_df.scenario)
fig, ax = plt.subplots(figsize=(12, 5), layout="constrained")
mesh = ax.imshow(matrix, aspect="auto", cmap="RdYlGn_r", vmin=0.7, vmax=1.2)
ax.set_xticks(
    np.arange(len(matrix.columns)), matrix.columns, rotation=35, ha="right", fontsize=8
)
ax.set_yticks(np.arange(len(matrix.index)), matrix.index, fontsize=9)
for i in range(len(matrix.index)):
    for j in range(len(matrix.columns)):
        ax.text(j, i, f"{matrix.iloc[i,j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(mesh, ax=ax, label="Healthy kill (%)")
ax.set_title("Selected designs under one-at-a-time model perturbations")
present_figure(fig)


### What the results support

- A single frozen model can generate density-specific designs with high tumour kill while meeting the nominal healthy-cell constraint at moderate and high densities.
- A design selected at one density need not remain feasible at another. Optimising again can restore the modelled selectivity.
- At very low density, the original payload limits efficacy. The 10 nM case is a separate potency intervention.
- Matching fitted uptake landmarks does not validate the model. The Figure 8 occupancy/uptake disagreement remains visible in the comparison table.
- Selected kinetics lie near the search boundaries. They are candidate designs within the declared domain, not unique or globally certified optima, and have no sequence-level feasibility assessment.
- Some perturbed physiological assumptions produce healthy kill above 1%. A future robust optimisation would need a prespecified uncertainty set.

## 9. Build the accompanying report

The following cell produces `report/report.tex` from the results just computed. It uses only the figures generated above. If `pdflatex` is available, it also builds `report/report.pdf` in a temporary compilation directory, leaving no auxiliary files in the repository. Python-only users can still run the entire scientific workflow and read every result in the notebook.